# STAT 486 - Deliverable 3: Supervised Modeling

### Load and Clean

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("/Users/lizethmendoza/Documents/STAT 486/STAT486FinalProject/data/Airline_Delay_Cause.csv")
df = df[df["arr_flights"] > 0].copy()

In [6]:
df.head(5)

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2023,8,9E,Endeavor Air Inc.,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",89.0,13.0,2.25,1.60,...,0.0,5.99,2.0,1.0,1375.0,71.0,761.0,118.0,0.0,425.0
1,2023,8,9E,Endeavor Air Inc.,ABY,"Albany, GA: Southwest Georgia Regional",62.0,10.0,1.97,0.04,...,0.0,7.42,0.0,1.0,799.0,218.0,1.0,62.0,0.0,518.0
2,2023,8,9E,Endeavor Air Inc.,AEX,"Alexandria, LA: Alexandria International",62.0,10.0,2.73,1.18,...,0.0,4.28,1.0,0.0,766.0,56.0,188.0,78.0,0.0,444.0
3,2023,8,9E,Endeavor Air Inc.,AGS,"Augusta, GA: Augusta Regional at Bush Field",66.0,12.0,3.69,2.27,...,0.0,1.57,1.0,1.0,1397.0,471.0,320.0,388.0,0.0,218.0
4,2023,8,9E,Endeavor Air Inc.,ALB,"Albany, NY: Albany International",92.0,22.0,7.76,0.00,...,0.0,11.28,2.0,0.0,1530.0,628.0,0.0,134.0,0.0,768.0


### Feature Engineering

In [3]:
df["carrier_rate"] = df["carrier_ct"] / df["arr_flights"]
df["weather_rate"] = df["weather_ct"] / df["arr_flights"]
df["nas_rate"] = df["nas_ct"] / df["arr_flights"]
df["security_rate"] = df["security_ct"] / df["arr_flights"]
df["late_aircraft_rate"] = df["late_aircraft_ct"] / df["arr_flights"]
df["cancel_rate"] = df["arr_cancelled"] / df["arr_flights"]
df["divert_rate"] = df["arr_diverted"] / df["arr_flights"]

In [4]:
# Binary target: high-delay month at this carrier-airport combination
df["delay_rate"] = df["arr_del15"] / df["arr_flights"]
df["high_delay"] = (df["delay_rate"] > 0.20).astype(int)

In [5]:
FEATURES = [
    "carrier_rate", "weather_rate", "nas_rate",
    "security_rate", "late_aircraft_rate",
    "cancel_rate", "divert_rate", "arr_flights"
]

### Dataset Summary

In [6]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total carrier-airport-month observations : {len(df)}")
print(f"High-delay observations (class = 1)      : {df['high_delay'].sum()} "
      f"({df['high_delay'].mean()*100:.1f}%)")
print(f"Low-delay observations  (class = 0)      : {(df['high_delay']==0).sum()} "
      f"({(df['high_delay']==0).mean()*100:.1f}%)")

DATASET SUMMARY
Total carrier-airport-month observations : 171426
High-delay observations (class = 1)      : 65043 (37.9%)
Low-delay observations  (class = 0)      : 106383 (62.1%)


### Train / Validation Split

In [7]:
X = df[FEATURES].fillna(0)
y = df["high_delay"]
 
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
 
print(f"\nTrain size : {len(X_train)} rows")
print(f"Val size   : {len(X_val)} rows")


Train size : 137140 rows
Val size   : 34286 rows


#### Model 1: Decision Tree (baseline)

In [8]:
dt = DecisionTreeClassifier(
    max_depth=4,         
    min_samples_leaf=5,   
    random_state=42
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_val)
y_prob_dt = dt.predict_proba(X_val)[:, 1]
 
acc_dt     = accuracy_score(y_val, y_pred_dt)
auc_dt     = roc_auc_score(y_val, y_prob_dt)
cv_auc_dt  = cross_val_score(dt, X_train, y_train, cv=5, scoring="roc_auc").mean()

#### Model 2: Random Forest (stronger)

In [9]:

rf = RandomForestClassifier(
    n_estimators=200,   
    max_depth=10,      
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)
y_prob_rf  = rf.predict_proba(X_val)[:, 1]
 
acc_rf     = accuracy_score(y_val, y_pred_rf)
auc_rf     = roc_auc_score(y_val, y_prob_rf)
cv_auc_rf  = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()

### Metrics Table

In [10]:
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(f"{'Model':<25} {'Val Accuracy':>14} {'Val AUC':>10} {'CV AUC (5-fold)':>16}")
print("-" * 67)
print(f"{'Decision Tree':<25} {acc_dt:>14.3f} {auc_dt:>10.3f} {cv_auc_dt:>16.3f}")
print(f"{'Random Forest':<25} {acc_rf:>14.3f} {auc_rf:>10.3f} {cv_auc_rf:>16.3f}")
 
print("\nDecision Tree — Classification Report:")
print(classification_report(y_val, y_pred_dt, target_names=["Low delay", "High delay"]))
 
print("Random Forest — Classification Report:")
print(classification_report(y_val, y_pred_rf, target_names=["Low delay", "High delay"]))



MODEL COMPARISON
Model                       Val Accuracy    Val AUC  CV AUC (5-fold)
-------------------------------------------------------------------
Decision Tree                      0.906      0.960            0.959
Random Forest                      0.981      0.999            0.999

Decision Tree — Classification Report:
              precision    recall  f1-score   support

   Low delay       0.91      0.94      0.93     21277
  High delay       0.89      0.85      0.87     13009

    accuracy                           0.91     34286
   macro avg       0.90      0.90      0.90     34286
weighted avg       0.91      0.91      0.91     34286

Random Forest — Classification Report:
              precision    recall  f1-score   support

   Low delay       0.98      0.99      0.98     21277
  High delay       0.98      0.97      0.97     13009

    accuracy                           0.98     34286
   macro avg       0.98      0.98      0.98     34286
weighted avg       0.98      

### Rules & Interpretability

In [11]:
print("Decision Tree Rules (depth ≤ 4):")
print(export_text(dt, feature_names=FEATURES, max_depth=4))

Decision Tree Rules (depth ≤ 4):
|--- late_aircraft_rate <= 0.07
|   |--- nas_rate <= 0.08
|   |   |--- carrier_rate <= 0.10
|   |   |   |--- carrier_rate <= 0.07
|   |   |   |   |--- class: 0
|   |   |   |--- carrier_rate >  0.07
|   |   |   |   |--- class: 0
|   |   |--- carrier_rate >  0.10
|   |   |   |--- carrier_rate <= 0.14
|   |   |   |   |--- class: 0
|   |   |   |--- carrier_rate >  0.14
|   |   |   |   |--- class: 1
|   |--- nas_rate >  0.08
|   |   |--- carrier_rate <= 0.05
|   |   |   |--- nas_rate <= 0.13
|   |   |   |   |--- class: 0
|   |   |   |--- nas_rate >  0.13
|   |   |   |   |--- class: 1
|   |   |--- carrier_rate >  0.05
|   |   |   |--- late_aircraft_rate <= 0.03
|   |   |   |   |--- class: 1
|   |   |   |--- late_aircraft_rate >  0.03
|   |   |   |   |--- class: 1
|--- late_aircraft_rate >  0.07
|   |--- carrier_rate <= 0.06
|   |   |--- nas_rate <= 0.05
|   |   |   |--- late_aircraft_rate <= 0.11
|   |   |   |   |--- class: 0
|   |   |   |--- late_aircraft_ra

In [12]:
perm = permutation_importance(
    rf, X_val, y_val, n_repeats=10, random_state=42, scoring="roc_auc"
)
 
imp_df = pd.DataFrame({
    "feature":         FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std
}).sort_values("importance_mean", ascending=False)
 
print("\nPermutation Importance (Random Forest, validation AUC drop):")
print(imp_df.to_string(index=False))


Permutation Importance (Random Forest, validation AUC drop):
           feature  importance_mean  importance_std
late_aircraft_rate         0.141628        0.001034
      carrier_rate         0.118340        0.000999
          nas_rate         0.111311        0.000552
      weather_rate         0.007145        0.000164
       arr_flights         0.000197        0.000016
     security_rate         0.000154        0.000014
       cancel_rate         0.000127        0.000019
       divert_rate         0.000027        0.000006


### Plot

In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#185FA5" if i < 3 else "#B5D4F4" for i in range(len(imp_df))]
ax.barh(
    imp_df["feature"][::-1],
    imp_df["importance_mean"][::-1],
    xerr=imp_df["importance_std"][::-1],
    color=colors[::-1], edgecolor="none", height=0.55
)
ax.set_xlabel("Mean decrease in AUC when feature is shuffled", fontsize=11)
ax.set_title(
    "Permutation importance — Random Forest\n"
    "Predicting high-delay carrier-airport-month combinations",
    fontsize=11, fontweight="normal"
)
ax.spines[["top", "right"]].set_visible(False)
ax.axvline(0, color="#888", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nPermutation importance plot saved → permutation_importance.png")


Permutation importance plot saved → permutation_importance.png


### Confusion Matrices

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model, preds, title in zip(
    axes,
    [dt, rf],
    [y_pred_dt, y_pred_rf],
    ["Decision Tree (baseline)", "Random Forest"]
):
    ConfusionMatrixDisplay.from_predictions(
        y_val, preds,
        display_labels=["Low delay", "High delay"],
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(title, fontsize=11, fontweight="normal")
 
plt.suptitle(
    "Confusion matrices — validation set",
    fontsize=12, fontweight="normal", y=1.02
)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("Confusion matrix plot saved → confusion_matrices.png")

Confusion matrix plot saved → confusion_matrices.png
